In [5]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable, Sequence, Optional

import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

# =========================
# CONFIG
# =========================
BASE_DIR = Path("/Data/in Use")
ROUNDS_PATH = BASE_DIR / "combined_rounds_all_2017_2026.csv"
SHORTLIST_2026_PATH = BASE_DIR / "preseason_shortlist_2026.csv"  # adjust if needed

AS_OF_DATE = "2026-01-01"   # preseason cutoff (exclusive)
WINDOWS = (40, 24, 12)
TOP_N = 50

STATS = [
    "sg_total",
    "sg_app",
    "sg_arg",
    "sg_putt",
    "driving_dist",
    "driving_acc",
    "round_score",
]

# Helpful to keep in the output if present on your shortlist
SHORTLIST_KEEP_COLS = ["dg_id", "player_name", "is_liv", "notes", "tag_event_1", "tag_event_2", "tag_event_3", "tag_event_4"]


# =========================
# HELPERS
# =========================
def _pick_date_col(df: pd.DataFrame) -> str:
    """Choose the best date column available in rounds data."""
    if "round_date" in df.columns:
        return "round_date"
    if "event_completed" in df.columns:
        return "event_completed"
    raise ValueError("rounds_df must have 'round_date' or 'event_completed'.")


def _ensure_datetime(df: pd.DataFrame, col: str) -> pd.DataFrame:
    out = df.copy()
    out[col] = pd.to_datetime(out[col], errors="coerce")
    return out


def _ensure_numeric(df: pd.DataFrame, cols: Iterable[str]) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out


def _sort_rounds_desc(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    """Sort rounds so head(w) means 'most recent w rounds'."""
    sort_cols = ["dg_id", date_col]
    asc = [True, False]

    if "event_id" in df.columns:
        df = df.copy()
        df["event_id"] = pd.to_numeric(df["event_id"], errors="coerce")
        sort_cols.append("event_id")
        asc.append(False)

    # Prefer round_num, else round
    if "round_num" in df.columns:
        df = df.copy()
        df["round_num"] = pd.to_numeric(df["round_num"], errors="coerce")
        sort_cols.append("round_num")
        asc.append(False)
    elif "round" in df.columns:
        df = df.copy()
        df["round"] = pd.to_numeric(df["round"], errors="coerce")
        sort_cols.append("round")
        asc.append(False)

    return df.sort_values(sort_cols, ascending=asc)


def _player_rolling_for_windows(
    g: pd.DataFrame,
    windows: Sequence[int],
    stats: Sequence[str],
) -> pd.Series:
    """
    For one player's rounds (already sorted most-recent-first),
    compute windowed means for each stat.
    """
    out: dict[str, float] = {}

    for w in windows:
        sub = g.head(w)
        for stat in stats:
            out[f"{stat}_L{w}"] = sub[stat].mean() if stat in sub.columns else np.nan

    # Diagnostics
    out["n_rounds_used"] = len(g)
    return pd.Series(out)


def compute_rolling_stats(
    rounds_df: pd.DataFrame,
    as_of_date: str | pd.Timestamp,
    dg_ids: Iterable[int],
    windows: Sequence[int] = (40, 24, 12),
    stats: Sequence[str] = STATS,
) -> pd.DataFrame:
    """
    Compute rolling window means for `stats` using rounds strictly before `as_of_date`.
    Output is wide: one row per dg_id, columns like stat_L40/stat_L24/stat_L12.
    """
    ts = pd.to_datetime(as_of_date, errors="coerce")
    if ts is pd.NaT:
        raise ValueError(f"Invalid as_of_date: {as_of_date}")

    if "dg_id" not in rounds_df.columns:
        raise ValueError("rounds_df missing required column: dg_id")

    df = rounds_df.copy()
    df["dg_id"] = pd.to_numeric(df["dg_id"], errors="coerce")

    dg_ids = [int(x) for x in dg_ids]
    df = df[df["dg_id"].isin(dg_ids)].copy()

    date_col = _pick_date_col(df)
    df = _ensure_datetime(df, date_col)

    # keep only history strictly before cutoff
    df = df[df[date_col] < ts].copy()
    if df.empty:
        return pd.DataFrame({"dg_id": dg_ids})

    df = _ensure_numeric(df, stats)
    df = _sort_rounds_desc(df, date_col)

    grouped = (
        df.groupby("dg_id", group_keys=False)
          .apply(_player_rolling_for_windows, windows=windows, stats=stats)
          .reset_index()
    )

    # Ensure all requested dg_ids appear even if no rounds
    grouped = pd.DataFrame({"dg_id": dg_ids}).merge(grouped, on="dg_id", how="left")

    return grouped


def make_readable_multilevel(
    df: pd.DataFrame,
    base_cols: Optional[Sequence[str]] = None,
) -> pd.DataFrame:
    """
    Convert flat columns like 'sg_app_L40' into MultiIndex columns:
        ('sg_app', 'L40')
    This is much easier to scan in notebooks.
    """
    if base_cols is None:
        base_cols = ["dg_id", "player_name", "is_liv", "n_rounds_used"]

    base_cols = [c for c in base_cols if c in df.columns]
    stat_cols = [c for c in df.columns if "_L" in c]

    tuples = []
    for c in stat_cols:
        stat, window = c.rsplit("_L", 1)
        tuples.append((stat, f"L{window}"))

    multi = pd.MultiIndex.from_tuples(tuples, names=["stat", "window"])

    stats_df = df[stat_cols].copy()
    stats_df.columns = multi

    base_df = df[base_cols].copy()
    return pd.concat([base_df, stats_df], axis=1)


def save_flat_csv_from_multilevel(df_multi: pd.DataFrame, out_path: Path) -> None:
    """
    Save MultiIndex columns to CSV by flattening them to 'stat__window' names.
    (CSV doesn't preserve MultiIndex well.)
    """
    df_flat = df_multi.copy()
    new_cols = []
    for c in df_flat.columns:
        if isinstance(c, tuple):
            new_cols.append(f"{c[0]}__{c[1]}")
        else:
            new_cols.append(str(c))
    df_flat.columns = new_cols
    df_flat.to_csv(out_path, index=False)


# =========================
# MAIN
# =========================
# Load data
rounds = pd.read_csv(ROUNDS_PATH, dtype={"round_date": "string"}, low_memory=False)
shortlist = pd.read_csv(SHORTLIST_2026_PATH)

# shortlist must have dg_id
if "dg_id" not in shortlist.columns:
    raise ValueError("shortlist file missing required column: dg_id")

shortlist["dg_id"] = pd.to_numeric(shortlist["dg_id"], errors="coerce")

top = (
    shortlist
    .dropna(subset=["dg_id"])
    .head(TOP_N)
    .copy()
)

dg_ids = top["dg_id"].astype(int).tolist()

# Compute rolling stats
roll = compute_rolling_stats(
    rounds_df=rounds,
    as_of_date=AS_OF_DATE,
    dg_ids=dg_ids,
    windows=WINDOWS,
    stats=STATS,
)

# Merge back onto shortlist (keep common/expected columns if they exist)
keep_cols = [c for c in SHORTLIST_KEEP_COLS if c in top.columns]
top_keep = top[keep_cols].copy()

out_wide = top_keep.merge(roll, on="dg_id", how="left")

# Build a readable MultiIndex version (best for notebooks)
out_readable = make_readable_multilevel(
    out_wide,
    base_cols=["dg_id", "player_name", "is_liv", "n_rounds_used"],
)

# Save outputs:
# 1) Wide/flat CSV (portable)
out_wide_path = BASE_DIR / f"shortlist_2026_top{TOP_N}_rolling_asof_{AS_OF_DATE}_wide.csv"
out_wide.to_csv(out_wide_path, index=False)

# 2) Readable (MultiIndex) saved as flattened "stat__window" for CSV compatibility
out_readable_path = BASE_DIR / f"shortlist_2026_top{TOP_N}_rolling_asof_{AS_OF_DATE}_readable.csv"
save_flat_csv_from_multilevel(out_readable, out_readable_path)

print(f"Saved wide:     {out_wide_path}")
print(f"Saved readable: {out_readable_path}")

# Quick peek
print("\nWIDE preview:")
print(out_wide.head(15))

print("\nREADABLE preview (MultiIndex columns):")
print(out_readable.head(15))


Saved wide:     /Users/joshmacbook/python_projects/OAD/Data/in Use/shortlist_2026_top50_rolling_asof_2026-01-01_wide.csv
Saved readable: /Users/joshmacbook/python_projects/OAD/Data/in Use/shortlist_2026_top50_rolling_asof_2026-01-01_readable.csv

WIDE preview:
    dg_id  is_liv  notes  tag_event_1  tag_event_2  tag_event_3  tag_event_4  \
0   18417   False    NaN          NaN          NaN          NaN          NaN   
1   19195    True    NaN          NaN          NaN          NaN          NaN   
2   19841    True    NaN          NaN          NaN          NaN          NaN   
3   10091   False    NaN          NaN          NaN          NaN          NaN   
4   12294   False    NaN          NaN          NaN          NaN          NaN   
5   19895   False    NaN          NaN          NaN          NaN          NaN   
6   14578   False    NaN          NaN          NaN          NaN          NaN   
7   24968   False    NaN          NaN          NaN          NaN          NaN   
8   22085   False  